[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sensioai/blog/blob/master/028_pytorch_nn/pytorch_nn.ipynb)

LABORATORIO 5.
# Pytorch - Redes Neuronales

ESTUDIANTE: VALDA PLAZA CAMILA MONSERRAT.
CARRERA: INGENIERÍA DE SISTEMAS.

LABORATORIO #5.
DATASET DEL LABORATORIO #4 ONEVSALL.

DATOS DE ESTUDIANTES DONDE NOS PREDICE SI UN ESTUDIANTE SE VA A GRADUAR,VA A ABANDONAR O SEGUIRÁ.

Se utilizó todo el cuadernillo visto en clases, donde estamos haciendo la comparativa de cada modelo, y se realizó varias pruebas donde se pudo notar a pesar de la cantidad de neuronas o capas.

Aquí se importa la librería torch, que es la base de PyTorch. Con esto ya se habilitan todas las funciones necesarias para crear tensores, definir modelos de redes neuronales y entrenarlos.

In [134]:
import torch

## Modelos secuenciales

In [135]:
#D_in, H, D_out = 36, 300, 3

#model = torch.nn.Sequential(
    #torch.nn.Linear(D_in, H),
    #torch.nn.ReLU(),
   # torch.nn.Linear(H, D_out),
#)

D_in, H1, H2, D_out = 36, 300, 180, 3

model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H1),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.3),
    torch.nn.Linear(H1, H2),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.3),
    torch.nn.Linear(H2, D_out),
)

Se definen las dimensiones de la red neuronal: 36 entradas (características), 100 neuronas en una capa oculta y 3 salidas (clases a predecir). Luego, se construye un modelo secuencial en PyTorch compuesto por dos capas lineales conectadas por una función de activación ReLU. El bloque comentado muestra otra arquitectura más profunda con dos capas ocultas adicionales, que podría usarse si se quisiera un modelo más complejo.

In [136]:
outputs = model(torch.randn(600, 36))
outputs.shape

torch.Size([600, 3])

Se imprime la primera predicción completa del modelo, que corresponde a un vector de tres valores llamados logits. Estos aún no son probabilidades, sino salidas lineales de la red antes de aplicar softmax.

In [137]:
print(outputs[0][:])

tensor([ 0.1209, -0.2038, -0.4168], grad_fn=<SliceBackward0>)


Se muestra la arquitectura del modelo creado en forma legible. Se pueden ver claramente las capas: primero una capa lineal que transforma 36 entradas en 100 neuronas, después una función de activación ReLU y finalmente otra capa lineal que transforma las 100 neuronas en 3 salidas.

In [138]:
model

Sequential(
  (0): Linear(in_features=36, out_features=300, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.3, inplace=False)
  (3): Linear(in_features=300, out_features=180, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.3, inplace=False)
  (6): Linear(in_features=180, out_features=3, bias=True)
)

In [139]:
model.to("cuda")

Sequential(
  (0): Linear(in_features=36, out_features=300, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.3, inplace=False)
  (3): Linear(in_features=300, out_features=180, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.3, inplace=False)
  (6): Linear(in_features=180, out_features=3, bias=True)
)

Se listan los archivos que hay en el directorio de trabajo. Esto permite comprobar qué archivos se encuentran disponibles en el entorno de Colab en ese momento.

In [140]:
import os
os.listdir()

['.config',
 'academic_data (3).csv',
 'academic_data (1).csv',
 'academic_data.csv',
 'academic_data (2).csv',
 'sample_data']

Aquí se prepara el dataset. Primero se carga un archivo desde la computadora del usuario hacia Colab utilizando la función files.upload(). Luego, se lee el archivo academic_data.csv con Pandas y se separan los datos en variables predictoras X y en la variable objetivo Y.

In [141]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

data = pd.read_csv("academic_data.csv")

X, Y = data.drop(columns=["Target"]), data["Target"]

print(X.shape, Y.shape)

Saving academic_data.csv to academic_data (4).csv
(76518, 36) (76518,)


Se define una función para normalizar los datos, restando la media y dividiendo entre la desviación estándar. Después de normalizar, los datos se dividen en entrenamiento (80%) y prueba (20%). También se ajustan las etiquetas de salida para que empiecen en 0 en lugar de 1. Finalmente, se muestran estadísticas como mínimos, máximos, media, desviación estándar y tamaño de los conjuntos, lo que confirma que la normalización se aplicó correctamente.

In [142]:
import numpy as np
def featureNormalize(X):
    X_norm = X.copy()
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    X_norm = (X - mu) / sigma
    return X_norm, mu, sigma

X_norm, mu, sigma = featureNormalize(X)

split = int(0.8 * len(X_norm))
X_train, X_test = X_norm[:split], X_norm[split:]
y_train, y_test = Y[:split], Y[split:]

y_train -= 1
y_test -= 1

print("X_train mínimo:", X_train.min(), "X_train máximo:", X_train.max())
print("X_train media:", X_train.mean(), "X_train std:", X_train.std())
print("Tamaño:", X_train.shape, y_train.shape)

X_train mínimo: Marital status                                   -0.253437
Application mode                                 -0.902423
Application order                                -1.337314
Course                                           -4.972914
Daytime/evening attendance                       -3.287603
Previous qualification                           -0.308308
Previous qualification (grade)                   -3.399536
Nacionality                                      -0.066801
Mother's qualification                           -1.223274
Father's qualification                           -1.502914
Mother's occupation                              -0.491269
Father's occupation                              -0.528580
Admission grade                                  -2.417081
Displaced                                        -1.149614
Educational special needs                        -0.061251
Debtor                                           -0.277252
Tuition fees up to date                 

In [143]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_train_bal, y_train_bal = ros.fit_resample(X_train, y_train)

In [144]:
import pandas as pd

pd.Series(y_train_bal).value_counts()

,count
Target,
0,29010
1,29010
2,29010


Se implementan manualmente dos funciones: la primera es softmax, que convierte los valores de salida (logits) en probabilidades, y la segunda es cross_entropy, que calcula la función de pérdida entre las predicciones del modelo y las etiquetas reales, midiendo así qué tan bien se está clasificando.

In [145]:
def softmax(x):
    return torch.exp(x) / torch.exp(x).sum(axis=-1,keepdims=True)

def cross_entropy(output, target):
    logits = output[torch.arange(len(output)), target]
    loss = - logits + torch.log(torch.sum(torch.exp(output), axis=-1))
    loss = loss.mean()
    return loss

In [146]:
# X_train

Se verifica si hay GPU disponible en el entorno de Colab. El resultado es True, lo que confirma que el entrenamiento puede acelerarse usando la tarjeta gráfica.

In [147]:
torch.cuda.is_available()

True

Se imprime el dataset original X para observar las primeras filas y columnas de los datos con sus 36 características. Esto sirve para visualizar el contenido real del archivo cargado y entender qué tipo de información se está usando en el entrenamiento.

In [148]:
print(X)

       Marital status  Application mode  Application order  Course  \
0                   1                 1                  1    9238   
1                   1                17                  1    9238   
2                   1                17                  2    9254   
3                   1                 1                  3    9500   
4                   1                 1                  2    9500   
...               ...               ...                ...     ...   
76513               1                17                  1    9254   
76514               1                 1                  6    9254   
76515               5                17                  1    9085   
76516               1                 1                  3    9070   
76517               1                 1                  1    9773   

       Daytime/evening attendance  Previous qualification  \
0                               1                       1   
1                               1    

Aquí comienza el entrenamiento del modelo. Primero se convierten los datos de entrenamiento en tensores de PyTorch y se copian en la GPU. Luego se configuran parámetros importantes: el número de épocas (350), la tasa de aprendizaje (0.8) y cada cuántas épocas se mostrará el error (cada 10). Durante el bucle de entrenamiento, en cada época se hace una pasada hacia adelante para obtener predicciones, se calcula la pérdida con la función cross_entropy, se obtienen gradientes con backpropagation y finalmente se actualizan los pesos del modelo de manera manual. A medida que avanza el entrenamiento, se imprime la pérdida promedio, mostrando cómo esta disminuye progresivamente.

In [149]:
X_t = torch.from_numpy(X_train.values).float().cuda()
Y_t = torch.from_numpy(y_train.values).long().cuda()

epochs = 2000
lr = 0.8
log_each = 10
l = []
for e in range(1, epochs + 1):

    y_pred = model(X_t)
    loss = cross_entropy(y_pred, Y_t)
    l.append(loss.item())
    model.zero_grad()
    loss.backward()

    with torch.no_grad():
        for param in model.parameters():
            param -= lr * param.grad

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

Epoch 10/2000 Loss 0.71005
Epoch 20/2000 Loss 0.67825
Epoch 30/2000 Loss 0.62404
Epoch 40/2000 Loss 0.59390
Epoch 50/2000 Loss 0.57463
Epoch 60/2000 Loss 0.56101
Epoch 70/2000 Loss 0.55085
Epoch 80/2000 Loss 0.54290
Epoch 90/2000 Loss 0.53647
Epoch 100/2000 Loss 0.53115
Epoch 110/2000 Loss 0.52663
Epoch 120/2000 Loss 0.52271
Epoch 130/2000 Loss 0.51934
Epoch 140/2000 Loss 0.51635
Epoch 150/2000 Loss 0.51369
Epoch 160/2000 Loss 0.51131
Epoch 170/2000 Loss 0.50915
Epoch 180/2000 Loss 0.50718
Epoch 190/2000 Loss 0.50535
Epoch 200/2000 Loss 0.50369
Epoch 210/2000 Loss 0.50213
Epoch 220/2000 Loss 0.50071
Epoch 230/2000 Loss 0.49939
Epoch 240/2000 Loss 0.49814
Epoch 250/2000 Loss 0.49695
Epoch 260/2000 Loss 0.49584
Epoch 270/2000 Loss 0.49480
Epoch 280/2000 Loss 0.49382
Epoch 290/2000 Loss 0.49290
Epoch 300/2000 Loss 0.49203
Epoch 310/2000 Loss 0.49119
Epoch 320/2000 Loss 0.49038
Epoch 330/2000 Loss 0.48961
Epoch 340/2000 Loss 0.48886
Epoch 350/2000 Loss 0.48816
Epoch 360/2000 Loss 0.48749
E

Ese código evalúa el desempeño del modelo en el conjunto de prueba midiendo el accuracy. Primero, con model.eval() se activa el modo evaluación para deshabilitar capas como dropout y garantizar predicciones estables. Luego, los datos se pasan al modelo para obtener los logits, que con softmax se convierten en probabilidades. Después, torch.argmax selecciona la clase con mayor probabilidad como predicción final. Finalmente, accuracy_score compara esas predicciones con las etiquetas reales y devuelve la proporción de aciertos, es decir, el porcentaje de muestras clasificadas correctamente.

In [150]:
from sklearn.metrics import accuracy_score

def evaluate(x):
    model.eval()
    y_pred = model(x)
    y_probas = softmax(y_pred)
    return torch.argmax(y_probas, axis=1)

y_pred = evaluate(torch.from_numpy(X_test.values).float().cuda())
accuracy_score(y_test.values, y_pred.cpu().numpy())

0.8186748562467329

## Optimizadores y Funciones de pérdida

La función de pérdida utilizada es CrossEntropyLoss, la cual compara las salidas del modelo con las etiquetas verdaderas y mide qué tan lejos están las predicciones de las respuestas correctas. Es la más común en clasificación multiclase porque aplica internamente el softmax y el log-loss, de modo que penaliza más fuerte las predicciones que se alejan mucho de la clase real.

In [151]:
criterion = torch.nn.CrossEntropyLoss()

El optimizador elegido es el descenso de gradiente estocástico (Stochastic Gradient Descent, SGD), que actualiza los pesos del modelo en cada paso de entrenamiento a partir de los gradientes calculados. Para controlar la magnitud de esos cambios se fija una tasa de aprendizaje de 0.8, que determina qué tan grandes son los ajustes realizados en cada iteración.

In [152]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.8)

Se define una red con 36 entradas, 100 neuronas en la capa oculta y 3 salidas, usando ReLU y se envía a la GPU. La pérdida se calcula con CrossEntropyLoss y los pesos se actualizan con SGD a tasa 0.8. Durante 1500 épocas, se realizan forward pass, cálculo de pérdida, retropropagación y actualización de pesos, registrando la pérdida periódicamente. Finalmente, el modelo se evalúa sobre los datos de prueba y se calcula el accuracy para medir la proporción de predicciones correctas.

In [153]:
D_in, H, D_out = 36, 100, 3

model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out),
).to("cuda")

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.8)

epochs = 1500
log_each = 10
l = []
model.train()
for e in range(1, epochs+1):
    y_pred = model(X_t)
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

y_pred = evaluate(torch.from_numpy(X_test.values).float().cuda())
accuracy_score(y_test.values, y_pred.cpu().numpy())

Epoch 10/1500 Loss 0.65934
Epoch 20/1500 Loss 0.58645
Epoch 30/1500 Loss 0.55508
Epoch 40/1500 Loss 0.53727
Epoch 50/1500 Loss 0.52554
Epoch 60/1500 Loss 0.51710
Epoch 70/1500 Loss 0.51064
Epoch 80/1500 Loss 0.50549
Epoch 90/1500 Loss 0.50126
Epoch 100/1500 Loss 0.49771
Epoch 110/1500 Loss 0.49467
Epoch 120/1500 Loss 0.49203
Epoch 130/1500 Loss 0.48971
Epoch 140/1500 Loss 0.48766
Epoch 150/1500 Loss 0.48581
Epoch 160/1500 Loss 0.48415
Epoch 170/1500 Loss 0.48264
Epoch 180/1500 Loss 0.48126
Epoch 190/1500 Loss 0.47999
Epoch 200/1500 Loss 0.47881
Epoch 210/1500 Loss 0.47772
Epoch 220/1500 Loss 0.47671
Epoch 230/1500 Loss 0.47576
Epoch 240/1500 Loss 0.47486
Epoch 250/1500 Loss 0.47403
Epoch 260/1500 Loss 0.47323
Epoch 270/1500 Loss 0.47248
Epoch 280/1500 Loss 0.47177
Epoch 290/1500 Loss 0.47110
Epoch 300/1500 Loss 0.47045
Epoch 310/1500 Loss 0.46984
Epoch 320/1500 Loss 0.46925
Epoch 330/1500 Loss 0.46868
Epoch 340/1500 Loss 0.46814
Epoch 350/1500 Loss 0.46762
Epoch 360/1500 Loss 0.46712
E

0.8163225300575013

## Modelos custom

Se define una clase que hereda de torch.nn.Module para crear un modelo personalizado. En el constructor se inicializan las capas: una lineal de entrada a oculta, la activación ReLU y otra lineal de oculta a salida. El método forward especifica cómo fluyen los datos a través de estas capas, aplicando primero la transformación lineal, luego la activación y finalmente la capa de salida para obtener las predicciones.

In [154]:

class ModeloPersonalizado(torch.nn.Module):
    def __init__(self, D_in, H, D_out):
        super(ModeloPersonalizado, self).__init__()
        self.fc1 = torch.nn.Linear(D_in, H)
        self.relu = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(H, D_out)
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

Se crea una instancia del modelo personalizado con 36 entradas, 100 neuronas en la capa oculta y 3 salidas. Luego se generan 500 datos de prueba aleatorios con 36 características cada uno y se pasan por el modelo para obtener las predicciones. Finalmente, se imprime la forma de la salida, confirmando que el modelo produce 500 predicciones con 3 valores cada una, correspondientes a las 3 clases.

In [155]:
model = ModeloPersonalizado(36, 100, 3)
x_prueba = torch.randn(500, 36)
print(x_prueba)
outputs = model(x_prueba)
outputs.shape

tensor([[ 1.2769, -0.2026,  0.1864,  ..., -2.1136,  0.5787, -1.6829],
        [-0.4108, -0.9325,  0.6501,  ..., -0.8472, -0.2165, -2.2687],
        [ 0.1225, -2.2441, -1.7398,  ...,  0.3715,  0.3292, -0.6781],
        ...,
        [ 1.1391,  0.4065, -1.0686,  ...,  0.1172, -0.3208,  1.0177],
        [-0.5162,  1.1109,  1.3756,  ...,  0.9279, -0.9026, -0.1616],
        [-0.3519, -1.0353, -2.1866,  ..., -0.2035, -0.3399,  1.9625]])


torch.Size([500, 3])

El modelo personalizado se envía a la GPU y se configura la función de pérdida CrossEntropyLoss junto con el optimizador SGD a tasa 0.8. Durante 100 épocas, se realiza el entrenamiento pasando los datos por la red, calculando la pérdida, retropropagando los gradientes y actualizando los pesos. Se registra la pérdida periódicamente para monitorear el aprendizaje. Finalmente, el modelo se evalúa en el conjunto de prueba y se calcula el accuracy para medir el porcentaje de predicciones correctas.

In [156]:
model.to("cuda")
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.8)

epochs = 100
log_each = 10
l = []
model.train()
for e in range(1, epochs+1):
    y_pred = model(X_t)
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

y_pred = evaluate(torch.from_numpy(X_test.values).float().cuda())
accuracy_score(y_test.values, y_pred.cpu().numpy())

Epoch 10/100 Loss 0.68348
Epoch 20/100 Loss 0.59836
Epoch 30/100 Loss 0.56275
Epoch 40/100 Loss 0.54285
Epoch 50/100 Loss 0.52990
Epoch 60/100 Loss 0.52066
Epoch 70/100 Loss 0.51365
Epoch 80/100 Loss 0.50809
Epoch 90/100 Loss 0.50356
Epoch 100/100 Loss 0.49976


0.815669106116048

Se define una clase de modelo personalizado que hereda de torch.nn.Module. En el constructor se crean dos capas lineales y una activación ReLU. En el método forward, los datos se transforman primero con la primera capa, luego se aplica ReLU, y finalmente la segunda capa recibe como entrada la suma de la salida de la activación y la primera transformación, introduciendo una especie de conexión residual que ayuda al flujo de gradientes durante el entrenamiento.

In [157]:
class ModelCustom2(torch.nn.Module):

    def __init__(self, D_in, H, D_out):
        super(ModelCustom2, self).__init__()
        self.fc1 = torch.nn.Linear(D_in, H)
        self.relu = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(H, D_out)

    def forward(self, x):
        x1 = self.fc1(x)
        x = self.relu(x1)
        x = self.fc2(x + x1)
        return x

Se instancia el modelo con conexión residual y se envía a la GPU. Se utiliza CrossEntropyLoss como función de pérdida y SGD con tasa de aprendizaje 0.01 para actualizar los pesos. Durante 100 épocas, se realizan forward pass, cálculo de pérdida, retropropagación y actualización de los parámetros, registrando periódicamente la pérdida promedio. Finalmente, el modelo se evalúa sobre los datos de prueba y se calcula el accuracy, mostrando la proporción de predicciones correctas.

In [158]:
model = ModelCustom2(36, 100, 3).to("cuda")
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

epochs = 100
log_each = 10
l = []
model.train()
for e in range(1, epochs+1):
    y_pred = model(X_t)
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

y_pred = evaluate(torch.from_numpy(X_test.values).float().cuda())
accuracy_score(y_test.values, y_pred.cpu().numpy())

Epoch 10/100 Loss 1.11380
Epoch 20/100 Loss 1.00034
Epoch 30/100 Loss 0.92580
Epoch 40/100 Loss 0.87302
Epoch 50/100 Loss 0.83349
Epoch 60/100 Loss 0.80256
Epoch 70/100 Loss 0.77755
Epoch 80/100 Loss 0.75677
Epoch 90/100 Loss 0.73916
Epoch 100/100 Loss 0.72398


0.7751568217459488

## Accediendo a las capas de una red

In [159]:
model

ModelCustom2(
  (fc1): Linear(in_features=36, out_features=100, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=100, out_features=3, bias=True)
)

In [160]:
model.fc1

Linear(in_features=36, out_features=100, bias=True)

También podemos acceder directamente a los tensores que contienen los parámetros con las propiedades adecuadas

In [161]:
model.fc1.weight

Parameter containing:
tensor([[ 0.0693, -0.0346,  0.0969,  ..., -0.0850,  0.1119,  0.0292],
        [ 0.0504,  0.0468,  0.0712,  ..., -0.1519,  0.1659,  0.1537],
        [ 0.0391, -0.0658,  0.1470,  ..., -0.0641,  0.0392, -0.0644],
        ...,
        [-0.1627, -0.1134, -0.0406,  ...,  0.1562, -0.0012,  0.0460],
        [-0.1177, -0.0733, -0.1544,  ..., -0.0819,  0.1131,  0.1557],
        [ 0.0455, -0.1585, -0.0296,  ..., -0.1050,  0.0006,  0.1148]],
       device='cuda:0', requires_grad=True)

In [162]:
model.fc1.bias

Parameter containing:
tensor([ 0.1430, -0.1210,  0.1580, -0.0098,  0.0781, -0.0829, -0.0745,  0.0749,
        -0.1344, -0.1157, -0.1077,  0.0207,  0.0343, -0.0376, -0.1226, -0.0429,
        -0.1362,  0.1439,  0.0016,  0.0541,  0.1359, -0.0280,  0.1778,  0.0076,
         0.1576,  0.0103, -0.0271,  0.1203, -0.0932, -0.1297, -0.1411,  0.0990,
         0.1667, -0.0011,  0.0491,  0.0378,  0.0482,  0.0078,  0.1118, -0.1556,
        -0.0782, -0.0675,  0.0486, -0.0795,  0.1379, -0.1002,  0.0255,  0.1261,
         0.1396, -0.0343,  0.1545, -0.0447,  0.0071, -0.1168,  0.0287,  0.1139,
        -0.1230,  0.0075, -0.0454,  0.1523,  0.1337, -0.0922, -0.0335,  0.0507,
        -0.0932,  0.0005, -0.1325, -0.1040, -0.1261, -0.0143,  0.0077, -0.1669,
         0.1255, -0.0323,  0.1318,  0.0140,  0.1161,  0.1593, -0.0550,  0.0488,
        -0.1508,  0.0468, -0.0381,  0.0769,  0.1666, -0.0260,  0.0536, -0.1614,
         0.0446,  0.0816,  0.0083, -0.1501, -0.0009,  0.0780, -0.1611,  0.0154,
        -0.0815, -

Es posible sobreescribir una capa de la siguiente manera

In [163]:
model.fc2 = torch.nn.Linear(100, 1)

model

ModelCustom2(
  (fc1): Linear(in_features=36, out_features=100, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=100, out_features=1, bias=True)
)

In [164]:
list(model.children())

[Linear(in_features=36, out_features=100, bias=True),
 ReLU(),
 Linear(in_features=100, out_features=1, bias=True)]

In [165]:
new_model = torch.nn.Sequential(*list(model.children())[:-2])
new_model

Sequential(
  (0): Linear(in_features=36, out_features=100, bias=True)
)

In [166]:
new_model = torch.nn.ModuleList(list(model.children())[:-1])
new_model

ModuleList(
  (0): Linear(in_features=36, out_features=100, bias=True)
  (1): ReLU()
)